<a href="https://colab.research.google.com/github/VivekReddy10-08-2004/USM_BAN_Big_Husky_Lovers/blob/main/Notebooks/BAN_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, r2_score

In [2]:
df = pd.read_csv("Cleaned_ML_Ready.csv")

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 216 entries, 0 to 215
Data columns (total 14 columns):
 #   Column                                 Non-Null Count  Dtype 
---  ------                                 --------------  ----- 
 0   JOB_REQUISITION                        216 non-null    object
 1   WORKER_SUB_TYPE                        216 non-null    object
 2   DAYS_OPEN                              216 non-null    int64 
 3   PRIMARY_LOCATION                       216 non-null    object
 4   JOB_PROFILE                            216 non-null    object
 5   MANAGEMENT_LEVEL_JOB_REQUISITION       216 non-null    object
 6   NUMBER_OF_OPENINGS_AVAILABLE           216 non-null    int64 
 7   JOB_DESCRIPTION                        216 non-null    object
 8   JOB_PROFILE_SUMMARY                    216 non-null    object
 9   CLEANED_JOB_DESCRIPTION                216 non-null    object
 10  CLEANED_JOB_PROFILE_SUMMARY            216 non-null    object
 11  PRIMARY_LOCATION_CO

NLP Vectorization

In [7]:
#TDIDF on Job descriptions
tfidf = TfidfVectorizer(max_features=50, stop_words ='english')
text_vectors = tfidf.fit_transform(df['CLEANED_JOB_DESCRIPTION'])
# Use PCA to compress those 50 text features into 3 core "themes" for easier clustering
pca = PCA(n_components=3)
text_pca = pca.fit_transform(text_vectors)
df['Text_Theme_1'] = text_pca[:, 0]
df['Text_Theme_2'] = text_pca[:, 1]
df['Text_Theme_3'] = text_pca[:, 2]

Clustering to find role complexity

In [8]:
# We cluster based on Location, Level, and the actual Text Themes of the job
cluster_features = ['PRIMARY_LOCATION_CODE', 'MANAGEMENT_LEVEL_JOB_REQUISITION_CODE',
                    'Text_Theme_1', 'Text_Theme_2', 'Text_Theme_3']
X_cluster = df[cluster_features]

# Grouping into 3 Tiers (e.g., Fast-Fill, Standard, Hard-to-Fill/Toxic)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['Complexity_Cluster'] = kmeans.fit_predict(X_cluster)

Regression model to predict the days open

In [12]:
# FIX 1: Use PCA text themes instead of the 50 raw TF-IDF columns to prevent overfitting
X_reg = df[['PRIMARY_LOCATION_CODE', 'WORKER_SUB_TYPE_CODE',
            'MANAGEMENT_LEVEL_JOB_REQUISITION_CODE', 'NUMBER_OF_OPENINGS_AVAILABLE',
            'Text_Theme_1', 'Text_Theme_2', 'Text_Theme_3']]

y_reg = df['DAYS_OPEN']

X_train, X_test, y_train, y_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# FIX 2: Constrain the Random Forest so it doesn't memorize outliers
rf_model = RandomForestRegressor(
    n_estimators=150,
    max_depth=5,              # Prevents trees from getting too complex
    min_samples_leaf=5,       # Requires at least 5 jobs in a leaf to make a rule
    random_state=42
)
rf_model.fit(X_train, y_train)

# Test accuracy
predictions_test = rf_model.predict(X_test)
new_r2 = r2_score(y_test, predictions_test)
new_mae = mean_absolute_error(y_test, predictions_test)

print(f"Tuned Model R^2 Score: {new_r2:.2f}")
print(f"Tuned Mean Absolute Error: {new_mae:.0f} days")

# Generate predictions
df['Predicted_Days_Open'] = np.round(rf_model.predict(X_reg), 0).astype(int)

Tuned Model R^2 Score: 0.16
Tuned Mean Absolute Error: 54 days


In [13]:
df['Complexity_Cluster'].value_counts()

,count
Complexity_Cluster,
0,111
2,59
1,46


Save dataset

In [15]:
output_file = 'Forecasted_HR_Data_For_PowerBI.csv'
df.to_csv(output_file, index=False)